# 3-Story Steel Frame v3 — Multi-Fidelity 3-Arm Benchmark (GPU)

Full 2D moment-resisting frame (Euler–Bernoulli beam-columns, actual column/beam
sections, SCWB plastic hinges), lateral load applied at **every floor**.

| Arm | Model | Inputs |
|---|---|---|
| **Arm 1** | low-fidelity, no training | `u_red` from `K = M Φ Ω² Φᵀ M` |
| **Arm 2** | purely data-driven | load + structural parameters only |
| **Arm 3** | multi-fidelity | load + `u_red` + `Δ_lin`, learns the residual `u_nl − u_red` |

**Split: 70 / 15 / 15.** Model selection on *validation*; every reported number is on
the held-out *test* set.

### Two corrections baked in (do not remove)
1. **No `physics_lowpass`.** Its 1.1 / 4.0 Hz bands were tuned for a wind turbine
   (f₁ = 0.27 Hz). This frame has f₁ = 1.37, f₂ = 4.38, f₃ = 8.25 Hz, so the filter cut
   the fundamental by 80% and modes 2–3 by 98% — and since it was applied to the
   *reconstructed* `u_red + Δ̂`, it damaged the baseline inside the prediction.
   Removing it moved Arm 3 from 0.4473 → 0.7611 with no retraining.
2. **Overlap-add hop `L_out/8`, no clip** — measured best.

**To run:** Runtime → Change runtime type → **GPU**, then upload
`gen_3story_frame_v3.py` when prompted.

In [ ]:
import os, sys, time, math, json
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU."
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

## 1. Data

The generator is pure numpy/scipy (no OpenSees) and takes ~5 min for N=1000.
Upload `gen_3story_frame_v3.py` when prompted, or set `USE_DRIVE = True` to load a
prebuilt `frame3_v3_database.npz` from Google Drive instead.

In [ ]:
USE_DRIVE = False          # True -> load a prebuilt .npz from Drive instead of generating
N_SAMPLES = 1000

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DB = '/content/drive/MyDrive/frame3_v3_database.npz'
else:
    if not os.path.exists('gen_3story_frame_v3.py'):
        from google.colab import files
        print("Upload gen_3story_frame_v3.py ...")
        files.upload()
    os.environ['MPLBACKEND'] = 'Agg'
    os.environ['N_SAMPLES']  = str(N_SAMPLES)
    t0 = time.time()
    rc = os.system(f'{sys.executable} -u gen_3story_frame_v3.py > gen.log 2>&1')
    print(f"generation rc={rc}  {time.time()-t0:.0f}s")
    print(open('gen.log').read()[-600:])
    DB = 'frame3_v3_database.npz'

print("database:", DB, f"{os.path.getsize(DB)/1e6:.0f} MB")

## 2. Consolidate + validity screen

Stack the per-floor series into `(N, 3, T)`. Drop samples whose nonlinear response
exceeds 1.5× the linear one — those are near-collapse, outside the intended
ductility range (target_mu ≤ 2.0), and they dominate any flattened metric.

In [ ]:
d = np.load(DB, allow_pickle=True)
cols = list(map(str, d['param_cols']))
g = np.where(d['converged'])[0]

u_nl  = np.stack([d[f'u{i}'][g]      for i in (1,2,3)], axis=1).astype(np.float32)
u_lin = np.stack([d[f'u{i}_lin'][g]  for i in (1,2,3)], axis=1).astype(np.float32)
u_red = np.stack([d[f'u{i}_mod'][g]  for i in (1,2,3)], axis=1).astype(np.float32)
q_sen = d['F'][g].astype(np.float32)
P     = d['params'][g].astype(np.float64)

amp  = np.max(np.abs(u_nl), axis=(1,2)) / (np.max(np.abs(u_red), axis=(1,2)) + 1e-30)
keep = amp < 1.5
print(f"validity screen: dropped {(~keep).sum()} / {len(g)} (amplification >= 1.5x)")
u_nl, u_lin, u_red, q_sen, P = u_nl[keep], u_lin[keep], u_red[keep], q_sen[keep], P[keep]

# (N,3,T) -> (N,T,3)
u_nl  = np.transpose(u_nl , (0,2,1))
u_lin = np.transpose(u_lin, (0,2,1))
u_red = np.transpose(u_red, (0,2,1))
q_sen = np.transpose(q_sen, (0,2,1))

N, T, S = u_nl.shape
dt = float(d['t'][1] - d['t'][0]); FS = 1.0/dt
fn = P[:, [cols.index(f'fn{i}') for i in (1,2,3)]]
print(f"N={N}  T={T}  S={S}  dt={dt:.3f}s  record={T*dt:.0f}s")
print(f"modal freqs [Hz]: f1={fn[:,0].mean():.2f}  f2={fn[:,1].mean():.2f}  f3={fn[:,2].mean():.2f}")

## 3. Split 70 / 15 / 15

In [ ]:
idx = np.random.default_rng(SEED).permutation(N)
n_tr, n_va = int(0.70*N), int(0.15*N)
idx_tr, idx_va, idx_te = idx[:n_tr], idx[n_tr:n_tr+n_va], idx[n_tr+n_va:]
assert len(set(idx_tr) & set(idx_va)) == 0 and len(set(idx_tr) & set(idx_te)) == 0
assert len(set(idx_va) & set(idx_te)) == 0
assert len(idx_tr)+len(idx_va)+len(idx_te) == N
print(f"train {len(idx_tr)}  val {len(idx_va)}  test {len(idx_te)}   (70/15/15)")

## 4. Metrics + Arm 1

`per-sample R²` is computed against each sample's own mean. Global R² pools every
sample against one grand mean, so it is dominated by between-sample amplitude
variance — both are reported.

In [ ]:
def metrics(true, pred, name):
    r2g = float(1 - ((true-pred)**2).sum()/(((true-true.mean())**2).sum()+1e-12))
    ps  = np.array([1 - ((true[i]-pred[i])**2).sum() /
                    (((true[i]-true[i].mean())**2).sum()+1e-30) for i in range(len(true))])
    pf  = [float(1 - ((true[:,:,s]-pred[:,:,s])**2).sum() /
           (((true[:,:,s]-true[:,:,s].mean())**2).sum()+1e-30)) for s in range(true.shape[2])]
    pk_t, pk_p = np.max(np.abs(true), axis=1), np.max(np.abs(pred), axis=1)
    pk   = float((np.abs(pk_t-pk_p)/(pk_t+1e-30)*100).mean())
    rmse = float(np.sqrt(np.mean((pred-true)**2))*1e3)
    return dict(name=name, R2=r2g, R2_persample=float(ps.mean()),
                R2_persample_med=float(np.median(ps)),
                R2_floor1=pf[0], R2_floor2=pf[1], R2_floor3=pf[2],
                peak_err_pct=pk, RMSE_mm=rmse)

def show(m):
    print(f"{m['name']:<26} R2={m['R2']:+.4f}  per-sample={m['R2_persample']:+.4f} "
          f"(med {m['R2_persample_med']:+.4f})  peak={m['peak_err_pct']:6.2f}%  "
          f"RMSE={m['RMSE_mm']:7.2f}mm")

RESULTS = []
m1 = metrics(u_nl[idx_te], u_red[idx_te], "Arm 1  low-fidelity")
RESULTS.append(m1); show(m1)
print(f"   per-floor: {m1['R2_floor1']:+.3f} / {m1['R2_floor2']:+.3f} / {m1['R2_floor3']:+.3f}")
print(f"   (u_lin for reference: R2={metrics(u_nl[idx_te], u_lin[idx_te],'x')['R2']:+.4f})")

## 5. Model — FiLM static encoder + bidirectional GRU + load skip

In [ ]:
L_IN = L_OUT = 200
L_ALL   = L_IN + L_OUT
STEP    = 100
HIDDEN  = 192
LAYERS  = 2
BATCH   = 64
LR      = 2e-3
EPOCHS  = 40
WD      = 4e-4
DROPOUT = 0.15

class FiLMStatic(nn.Module):
    def __init__(self, n_static, n_time, dropout=0.13):
        super().__init__()
        h = max(n_static*2, 64)
        self.film = nn.Sequential(nn.Linear(n_static, h), nn.ReLU(), nn.Linear(h, n_time*2))
        self.proj = nn.Sequential(nn.Linear(n_static, L_ALL), nn.Tanh(), nn.Linear(L_ALL, L_ALL))
        self.drop = nn.Dropout(dropout)
    def forward(self, x_time, x_static):
        s = x_static.squeeze(1)
        gam, beta = self.film(s).unsqueeze(1).chunk(2, dim=-1)
        out = (1.0 + gam) * x_time + beta
        sc  = self.drop(self.proj(s)).unsqueeze(2)
        return torch.cat([out, sc], dim=2)

class Represent(nn.Module):
    def __init__(self, in_size, hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT):
        super().__init__()
        self.gru = nn.GRU(in_size, hidden, num_layers=layers, batch_first=True,
                          bidirectional=True, dropout=dropout if layers > 1 else 0.0)
        for n_, p_ in self.gru.named_parameters():
            if   'weight_ih' in n_: nn.init.xavier_uniform_(p_)
            elif 'weight_hh' in n_: nn.init.orthogonal_(p_)
            elif 'bias'      in n_: nn.init.zeros_(p_)
        self.out_dim = hidden*2
    def forward(self, x):
        return self.gru(x)[0][:, -L_OUT:, :]

class Head(nn.Module):
    def __init__(self, in_dim, n_sensor):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 64), nn.GELU())
        self.out    = nn.Linear(64, n_sensor, bias=False)
        nn.init.xavier_uniform_(self.shared[0].weight, gain=0.1)
        nn.init.zeros_(self.out.weight)          # start at Delta = 0  => MF == Arm 1
    def forward(self, f_): return self.out(self.shared(f_))

class LoadSkip(nn.Module):
    # Per-sensor gain from statics x local load. Uses no fidelity information.
    def __init__(self, n_static, n_sensor):
        super().__init__()
        self.gain = nn.Sequential(nn.Linear(n_static, 64), nn.Tanh(), nn.Linear(64, n_sensor))
        nn.init.zeros_(self.gain[-1].weight); nn.init.zeros_(self.gain[-1].bias)
    def forward(self, q_out, s_flat):
        return q_out * self.gain(s_flat).unsqueeze(1)

## 6. Channels

* **Arm 3** — `q | u_red | Δ_lin | u_prev | sin | cos`, target `(u_nl − u_red)/σ`
* **Arm 2** — `q | u_prev | sin | cos`, target `u_nl/σ`, no baseline added back

`σ` is a **global per-floor** scale computed on the **train split only**.

In [ ]:
q_mu, q_sd = q_sen.mean(1, keepdims=True), q_sen.std(1, keepdims=True) + 1e-12
q_norm = ((q_sen - q_mu) / q_sd).astype(np.float32)
sig_red = u_red.std(1, keepdims=True) + 1e-12
u_red_n = (u_red / sig_red).astype(np.float32)
dlin_n  = ((u_lin - u_red) / sig_red).astype(np.float32)

t_loc = np.arange(L_ALL, dtype=np.float32) * dt
ph_sin = np.stack([np.sin(2*np.pi*float(fn[i,0])*t_loc) for i in range(N)]).astype(np.float32)
ph_cos = np.stack([np.cos(2*np.pi*float(fn[i,0])*t_loc) for i in range(N)]).astype(np.float32)

def logz(x, ref):
    lx = np.log(np.clip(x, 1e-30, None))
    return ((lx - lx[ref].mean(0)) / (lx[ref].std(0) + 1e-8)).astype(np.float32)

static = np.concatenate([logz(P, idx_tr), logz(fn, idx_tr),
                         logz(np.max(np.abs(q_sen), axis=1) + 1.0, idx_tr)], axis=1)
N_STATIC = static.shape[1]

resid = (u_nl - u_red).astype(np.float32)
SIG_R3 = resid[idx_tr].std(axis=(0,1), keepdims=True) + 1e-12     # Arm 3 target scale
SIG_R2 = u_nl [idx_tr].std(axis=(0,1), keepdims=True) + 1e-12     # Arm 2 target scale
print(f"N_STATIC={N_STATIC}   sigma_arm3(mm)={np.round(SIG_R3.ravel()*1e3,2)}   "
      f"sigma_arm2(mm)={np.round(SIG_R2.ravel()*1e3,2)}")

def starts():
    return [s for s in range(0, T, STEP) if s + L_ALL <= T + L_IN]

def build(sample_idx, arm, flip_aug=True):
    Xs, Ss, Ys = [], [], []
    tgt_all = (resid/SIG_R3) if arm == 3 else (u_nl/SIG_R2)
    z = np.zeros((L_IN, S), np.float32)
    for i in sample_idx:
        for sgn in ((1.0, -1.0) if flip_aug else (1.0,)):
            qp = np.concatenate([z, sgn*q_norm[i]])
            tp = np.concatenate([z, sgn*tgt_all[i]])
            bp = np.concatenate([z, sgn*u_red_n[i]])
            dp = np.concatenate([z, sgn*dlin_n[i]])
            for st in starts():
                en = st + L_ALL
                up = np.concatenate([tp[st:st+L_IN], np.zeros((L_OUT, S), np.float32)])
                ch = [qp[st:en]] + ([bp[st:en], dp[st:en]] if arm == 3 else [])
                ch += [up, ph_sin[i][:,None], ph_cos[i][:,None]]
                Xs.append(np.concatenate(ch, axis=1))
                Ss.append(static[i]); Ys.append(tp[st+L_IN:en])
    return (np.asarray(Xs, np.float32), np.asarray(Ss, np.float32)[:,None,:],
            np.asarray(Ys, np.float32))

class WinDS(Dataset):
    def __init__(self, X, Sx, Y): self.X, self.S, self.Y = map(torch.from_numpy, (X, Sx, Y))
    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i], self.S[i], self.Y[i]

## 7. Train + free-running evaluation

Best epoch by **validation** loss. Reconstruction uses overlap-add with hop
`L_out/8`, Tukey(0.3) taper, **no clip and no low-pass filter**.

In [ ]:
def tukey(n, alpha=0.3):
    w = np.ones(n, np.float32); k = int(alpha*n/2)
    if k > 0:
        t_ = 0.5*(1-np.cos(np.pi*np.arange(k)/k)); w[:k] = t_; w[-k:] = t_[::-1]
    return w

@torch.no_grad()
def rollout(mods, sample_idx, arm, hop_div=8):
    film, rep, head, skip = mods
    for m_ in mods: m_.eval()
    hop, tap = max(1, L_OUT//hop_div), tukey(L_OUT)
    out = np.zeros((len(sample_idx), T, S), np.float32)
    for b0 in range(0, len(sample_idx), 32):
        ch_i = sample_idx[b0:b0+32]; nb = len(ch_i)
        z = np.zeros((nb, L_IN, S), np.float32)
        qp = np.concatenate([z, q_norm[ch_i]], axis=1)
        bp = np.concatenate([z, u_red_n[ch_i]], axis=1)
        dp = np.concatenate([z, dlin_n [ch_i]], axis=1)
        acc = np.zeros((nb, T+L_IN, S), np.float32)
        wgt = np.zeros((nb, T+L_IN, 1), np.float32)
        est = np.zeros((nb, T+L_IN, S), np.float32)
        sb  = torch.from_numpy(static[ch_i][:,None,:]).to(device)
        for st in range(0, T, hop):
            en = st + L_ALL
            if en > T + L_IN: break
            up = np.concatenate([est[:, st:st+L_IN], np.zeros((nb, L_OUT, S), np.float32)], axis=1)
            ch = [qp[:, st:en]] + ([bp[:, st:en], dp[:, st:en]] if arm == 3 else [])
            ch += [up, np.repeat(ph_sin[ch_i][:,:,None], 1, 2), np.repeat(ph_cos[ch_i][:,:,None], 1, 2)]
            xb = torch.from_numpy(np.concatenate(ch, axis=2)).to(device)
            sf = sb.squeeze(1)
            p  = head(rep(film(xb, sb))) + skip(xb[:, -L_OUT:, :S], sf)
            p  = p.cpu().numpy()
            acc[:, st+L_IN:en] += p * tap[None,:,None]
            wgt[:, st+L_IN:en] += tap[None,:,None]
            sw = np.where(wgt[:, st+L_IN:en] > 0, wgt[:, st+L_IN:en], 1.0)
            est[:, st+L_IN:en] = acc[:, st+L_IN:en] / sw          # no clip
        sw = np.where(wgt > 0, wgt, 1.0)
        out[b0:b0+nb] = (acc/sw)[:, L_IN:L_IN+T]
    if arm == 3:
        return u_red[sample_idx] + out*SIG_R3                     # no low-pass filter
    return out*SIG_R2

def run_arm(arm, tag):
    Xtr, Str, Ytr = build(idx_tr, arm, True)
    Xva, Sva, Yva = build(idx_va, arm, False)
    n_time = Xtr.shape[2]
    print(f"[{tag}] windows train={len(Xtr)} val={len(Xva)}  channels={n_time}")
    film = FiLMStatic(N_STATIC, n_time).to(device)
    rep  = Represent(n_time+1).to(device)
    head = Head(rep.out_dim, S).to(device)
    skip = LoadSkip(N_STATIC, S).to(device)
    mods = (film, rep, head, skip)
    par  = [p for m_ in mods for p in m_.parameters()]
    print(f"[{tag}] params={sum(p.numel() for p in par):,}")

    dl_tr = DataLoader(WinDS(Xtr,Str,Ytr), batch_size=BATCH, shuffle=True,  drop_last=True)
    dl_va = DataLoader(WinDS(Xva,Sva,Yva), batch_size=BATCH)
    opt   = torch.optim.AdamW(par, lr=LR, weight_decay=WD)
    sch   = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=EPOCHS,
                                                steps_per_epoch=len(dl_tr), pct_start=0.3)
    crit, best, best_state = nn.MSELoss(), float('inf'), None
    t0 = time.time()
    for ep in range(1, EPOCHS+1):
        for m_ in mods: m_.train()
        for xb, sb, yb in dl_tr:
            xb, sb, yb = xb.to(device,non_blocking=True), sb.to(device), yb.to(device)
            opt.zero_grad()
            p = head(rep(film(xb, sb))) + skip(xb[:, -L_OUT:, :S], sb.squeeze(1))
            loss = crit(p, yb); loss.backward()
            nn.utils.clip_grad_norm_(par, 1.0); opt.step(); sch.step()
        for m_ in mods: m_.eval()
        tot = 0.0
        with torch.no_grad():
            for xb, sb, yb in dl_va:
                xb, sb, yb = xb.to(device), sb.to(device), yb.to(device)
                p = head(rep(film(xb, sb))) + skip(xb[:, -L_OUT:, :S], sb.squeeze(1))
                tot += crit(p, yb).item()*len(xb)
        va = tot/len(dl_va.dataset)
        if va < best:
            best = va
            best_state = [{k: v.detach().cpu().clone() for k, v in m_.state_dict().items()}
                          for m_ in mods]
        if ep % 5 == 0 or ep == 1:
            print(f"[{tag}] epoch {ep:3d}  val_mse={va:.5f}  ({time.time()-t0:.0f}s)")
    for m_, st_ in zip(mods, best_state): m_.load_state_dict(st_)
    print(f"[{tag}] best val_mse={best:.5f}  train {time.time()-t0:.0f}s")
    return mods

## 8. Arm 3 — multi-fidelity

In [ ]:
mods3 = run_arm(3, "ARM3")

# sanity: an untrained (zero-init) head must reproduce Arm 1 exactly
chk = FiLMStatic(N_STATIC, 4*S+2).to(device), Represent(4*S+3).to(device), \
      Head(HIDDEN*2, S).to(device), LoadSkip(N_STATIC, S).to(device)
p0 = rollout(chk, idx_te[:16], 3)
print(f"zero-init check |pred - u_red| = {np.abs(p0 - u_red[idx_te[:16]]).max():.2e} (expect ~0)")

pred3 = rollout(mods3, idx_te, 3)
m3 = metrics(u_nl[idx_te], pred3, "Arm 3  multi-fidelity"); RESULTS.append(m3); show(m3)
print(f"   per-floor: {m3['R2_floor1']:+.3f} / {m3['R2_floor2']:+.3f} / {m3['R2_floor3']:+.3f}")

## 9. Arm 2 — purely data-driven

In [ ]:
mods2 = run_arm(2, "ARM2")
pred2 = rollout(mods2, idx_te, 2)
m2 = metrics(u_nl[idx_te], pred2, "Arm 2  data-driven"); RESULTS.insert(1, m2); show(m2)
print(f"   per-floor: {m2['R2_floor1']:+.3f} / {m2['R2_floor2']:+.3f} / {m2['R2_floor3']:+.3f}")

## 10. α-blend (optional)

`u_MF = u_red + α·Δ̂`. α is fitted on **validation** and applied to **test**, so it is
an honest held-out number. α → 0 recovers Arm 1 exactly, which makes
"multi-fidelity never worse than its own baseline" a structural guarantee.

In [ ]:
pv = rollout(mods3, idx_va, 3)
dv, dt_ = pv - u_red[idx_va], pred3 - u_red[idx_te]
grid = np.linspace(0, 1.2, 25)
r2v = [1 - ((u_nl[idx_va]-(u_red[idx_va]+a*dv))**2).sum() /
       (((u_nl[idx_va]-u_nl[idx_va].mean())**2).sum()+1e-12) for a in grid]
a_star = float(grid[int(np.argmax(r2v))])
mA = metrics(u_nl[idx_te], u_red[idx_te] + a_star*dt_, f"Arm 3  alpha={a_star:.2f} (val-fit)")
RESULTS.append(mA)
print(f"alpha* fitted on val = {a_star:.2f}   (alpha=1 is the unfitted default)")
show(mA)

## 11. Summary — all metrics on the held-out test set

In [ ]:
import csv as _csv
hdr = ["name","R2","R2_persample","R2_persample_med","R2_floor1","R2_floor2","R2_floor3",
       "peak_err_pct","RMSE_mm"]
print(f"\n{'':<30}{'R2':>9}{'R2/sample':>11}{'peak %':>9}{'RMSE mm':>10}")
print("-"*69)
for m_ in RESULTS:
    print(f"{m_['name']:<30}{m_['R2']:>9.4f}{m_['R2_persample']:>11.4f}"
          f"{m_['peak_err_pct']:>9.2f}{m_['RMSE_mm']:>10.2f}")
with open('frame3_v3_test_metrics.csv','w',newline='') as f_:
    w_ = _csv.DictWriter(f_, fieldnames=hdr); w_.writeheader()
    for m_ in RESULTS: w_.writerow({k: m_[k] for k in hdr})
print("\nwrote frame3_v3_test_metrics.csv")
print(f"split: train {len(idx_tr)} / val {len(idx_va)} / test {len(idx_te)}  "
      f"(all numbers above are TEST)")